In [ ]:

# ============ IMPORTS ============
import pandas as pd
import numpy as np
import sys
from scipy.signal import find_peaks
from src.triple_barrier import triple_barrier
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# ========== CONFIG ==========
window = 7  # days
profit_target = 1.7 # profit_target times volatility for profit-taking
stop_loss = 1.2 # stop_loss times volatility for stop-loss
min_return_threshold = 0.005  # Anything above this threshold will be marked as +1. Currently disabled in triple_barrier.py
# ============================

# ============ LOAD CACHED DATA ============
engineered_features_cache = pd.read_pickle('data_cache/engineered_features.pkl')
optuna_trials_data = pd.read_pickle('data_cache/optuna_trials_data.pkl')
bitcoin_price_and_features = engineered_features_cache['bitcoin_price_and_features']
bitcoin_features = engineered_features_cache['bitcoin_features']

# Start with absolute features
labelled_df = bitcoin_features.copy()

# Add Close price for reference
labelled_df['Close'] = bitcoin_price_and_features['Close']

In [ ]:
# ============ PEAK DETECTION & LABELING ============
# Detect peaks
# peak_indices, properties = find_peaks(
#     labelled_df['Close'].values,
#     prominence = labelled_df['Close'].mean() * 0.10,
#     distance=30
# )

# # Initialize 'Near_Peak' column
# labelled_df['Near_Peak'] = 0

# # Label near-peak regions
# WINDOW_DAYS = 15
# for peak_idx in peak_indices:
#     start = max(0, peak_idx - WINDOW_DAYS)
#     end = min(len(labelled_df), peak_idx + WINDOW_DAYS)
#     labelled_df.iloc[start:end, labelled_df.columns.get_loc('Near_Peak')] = 1

In [ ]:
# ============ TRIPLE BARRIER LABELLING ============
triple_barrier_df = triple_barrier(
    price_series = labelled_df['Close'],
    volatility_series = labelled_df['Volatility_EWMA'],
    holding_period = window,
    profit_mult = profit_target,
    stop_mult = stop_loss,
    min_ret_threshold = min_return_threshold
)
labelled_df['Exit_Price'] = triple_barrier_df['exit_prices']
labelled_df[f'Return_{window}day'] = triple_barrier_df['returns']
labelled_df['Barrier_Hit_Day'] = triple_barrier_df['barrier_hit_times']
labelled_df[f'Label_{window}day'] = triple_barrier_df['labels']
# Drop nan values
labelled_df = labelled_df.dropna()

# Check for gaps in dates
missing_dates = pd.date_range(start=min(labelled_df.index),end=max(labelled_df.index)).difference(labelled_df.index)
print(f'Missing dates = {missing_dates}')

In [ ]:
# ============ SPLIT DATA ============
split_idx = int(len(labelled_df) * 0.8)
labelled_df["Set"] = "Train"
labelled_df.loc[labelled_df.index[split_idx:], "Set"] = "Test"
labelled_df.loc[labelled_df.index[split_idx:split_idx + window - 1], "Set"] = "Gap"

# Define X, y
target_col = f"Label_{window}day"
return_col = f"Return_{window}day"
drop_cols = [target_col, "Close","Near_Peak","Exit_Price",f"Label_{window}day","Barrier_Hit_Day","Set",return_col]  # keep only features

feature_cols = [col for col in labelled_df.columns if col not in drop_cols]

X_train = labelled_df.loc[labelled_df["Set"] == "Train", feature_cols]
y_train = labelled_df.loc[labelled_df["Set"] == "Train", target_col]
returns_train = labelled_df.loc[labelled_df["Set"] == "Train",return_col]
X_test = labelled_df.loc[labelled_df["Set"] == "Test", feature_cols]
y_test = labelled_df.loc[labelled_df["Set"] == "Test", target_col]
returns_test = labelled_df.loc[labelled_df["Set"] == "Test",return_col]

In [ ]:
# ============ SCALE BOUNDED DATA ============
bounded_features_and_limits = {
    "RSI":      (0,    100),
    "STOCHRSI": (0,    100),
    "STOCH_K":  (0,    100),
    "STOCH_D":  (0,    100),
    "WILLR":    (-100, 0),
    "MFI":      (0,    100),
    "ADX":      (0,    100),
    "ADXR":     (0,    100),
    "PLUS_DI":  (0,    100),
    "MINUS_DI": (0,    100),
    "AROONOSC": (-100, 100),
    "ULTOSC":   (0,    100),
    "CMO":      (-100, 100),
}

# Custom min-max scaler
for col, (lo, hi) in bounded_features_and_limits.items():
    X_train[col] = ((X_train[col] - lo) / (hi - lo)).clip(0,1)
    X_test[col]  = ((X_test[col]  - lo) / (hi - lo)).clip(0,1)

In [ ]:
# =========== CACHE LABELLED DATA ============
pd.to_pickle({
    'labelled_df':labelled_df,
    'X_train':X_train,
    'X_test':X_test,
    'y_train':y_train,
    'y_test':y_test,
    'returns_train': returns_train,
    'returns_test': returns_test,
    'profit_target':profit_target,
    'stop_loss':stop_loss
}, 'data_cache/labelled_data.pkl')